In [22]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time 
from datetime import timedelta

In [23]:
DATASET = "../dataset/splitted"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 6
EPOCHS = 10
BATCH_SIZE = 16
IMGSZ = 224
LR = 1e-4

In [24]:
transform_train = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

In [25]:
transform_test = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

In [26]:
train_dataset = datasets.ImageFolder(f"{DATASET}/train", transform=transform_train)
test_dataset  = datasets.ImageFolder(f"{DATASET}/test",  transform=transform_test)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

In [27]:
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
model = model.to(DEVICE)

In [28]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
criterion = nn.CrossEntropyLoss()

In [29]:
best_acc = 0.0

start_time = time.time()
print(f"Training started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_acc = correct / total

    # Evaluation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(test_loader):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "vit_best.pt")
        print(f"  ✅ Saved best model with val acc: {best_acc:.4f}")

end_time = time.time()
elapsed = timedelta(seconds=int(end_time - start_time))
print(f"\nTraining ended at:   {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total training time: {elapsed}")
print(f"Best Val Accuracy:   {best_acc:.4f}")
print(f"\nBest Val Accuracy: {best_acc:.4f}")

Training started at: 2026-03-07 10:39:32
Epoch [1/10] Train Loss: 0.1310 | Train Acc: 0.9533 | Val Loss: 0.0187 | Val Acc: 0.9956
  ✅ Saved best model with val acc: 0.9956
Epoch [2/10] Train Loss: 0.0584 | Train Acc: 0.9824 | Val Loss: 0.0376 | Val Acc: 0.9878
Epoch [3/10] Train Loss: 0.0234 | Train Acc: 0.9919 | Val Loss: 0.0009 | Val Acc: 1.0000
  ✅ Saved best model with val acc: 1.0000
Epoch [4/10] Train Loss: 0.0126 | Train Acc: 0.9967 | Val Loss: 0.0144 | Val Acc: 0.9967
Epoch [5/10] Train Loss: 0.0254 | Train Acc: 0.9924 | Val Loss: 0.0331 | Val Acc: 0.9889
Epoch [6/10] Train Loss: 0.0448 | Train Acc: 0.9843 | Val Loss: 0.0056 | Val Acc: 0.9989
Epoch [7/10] Train Loss: 0.0310 | Train Acc: 0.9905 | Val Loss: 0.0017 | Val Acc: 1.0000
Epoch [8/10] Train Loss: 0.0014 | Train Acc: 1.0000 | Val Loss: 0.0006 | Val Acc: 1.0000
Epoch [9/10] Train Loss: 0.0024 | Train Acc: 0.9995 | Val Loss: 0.0006 | Val Acc: 1.0000
Epoch [10/10] Train Loss: 0.0006 | Train Acc: 1.0000 | Val Loss: 0.0005 | 